# LaGCORL agent training: one workflow, selectable benchmark

Choose `BENCHMARK` once near the beginning (`"maxcut"` or `"vmp"`). The notebook then runs the same sequence of experiments for either benchmark: discrete PPO, GAE/VGAE training, GNN observations with discrete PPO and padded actions, iterative iDQN, and projection PPO. Benchmark-specific environment construction is encapsulated in `notebook_helpers_api.py`; the experiment sections do not branch on the benchmark.


In [ ]:
# Experiment controls — choose the benchmark here, once.
BENCHMARK = "maxcut"  # "maxcut" or "vmp"

## 0. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

%matplotlib inline

import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")

import matplotlib.pyplot as plt

plt.ioff()

In [ ]:
import sys
from pathlib import Path
import random
import shutil
from datetime import datetime
import numpy as np
import pandas as pd
from IPython.display import display
from stable_baselines3.common.vec_env import DummyVecEnv

cwd = Path.cwd().resolve()
if (cwd / "gae" / "train_gae.py").exists():
    PROJECT_ROOT = cwd
elif (cwd / "LaGCORL" / "gae" / "train_gae.py").exists():
    PROJECT_ROOT = cwd / "LaGCORL"
else:
    raise FileNotFoundError("Could not locate LaGCORL/gae/train_gae.py; set PROJECT_ROOT manually.")

for entry in (str(PROJECT_ROOT.parent), str(PROJECT_ROOT)):
    if entry not in sys.path:
        sys.path.insert(0, entry)
NOTEBOOK_DIR = cwd
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from LaGCORL.agents.notebook_train_api import train_agent_on_env
from LaGCORL.agents.notebook_test_api import test_agent_on_env
from LaGCORL.agents.notebook_train_gae_api import (
    train_gae_for_embeddings,
    load_gae_encoders,
    make_agent_training_wrapper_factory,
)
from notebook_helpers_api import (
    plot_tensorboard,
    save_agent_artifact,
    sample_benchmark_corpus,
    load_benchmark_env,
    create_run_layout,
    benchmark_on_corpus,
    save_training_metadata,
    save_metric_frame,
    build_padded_gnn_discrete_corpus,
    evaluate_padded_gnn_policy,
)

LOG_ROOT = PROJECT_ROOT.parent / "logs"
LOG_ROOT.mkdir(parents=True, exist_ok=True)

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
print("Project root:", PROJECT_ROOT)
print("Artifact root:", LOG_ROOT.resolve())

## 1. Select the benchmark and generate its randomized corpus

The corpus API creates the appropriate native environments, which are then used subsequently for training and evaluation. The first instance is fixed for training; all remaining instances are held out for evaluation.


In [ ]:
SAMPLED_SCENARIO_COUNT = 20
CORPUS_SEED = SEED
MAX_SWEEPS = 100

# MaxCut Params
GRAPH_SIZE_RANGE = (10, 20)       
EDGE_WEIGHT_RANGE = (10, 20)   

# VMP Params
VMP_RANGE = {
    "min_VMs": 20,
    "max_VMs": 60,
    "min_PMs": 40,
    "max_PMs": 80,
}

corpus = sample_benchmark_corpus(
    PROJECT_ROOT,
    BENCHMARK,
    count=SAMPLED_SCENARIO_COUNT,
    seed=CORPUS_SEED,
    max_sweeps=MAX_SWEEPS,
    graph_size_range=GRAPH_SIZE_RANGE,
    edge_weight_range=EDGE_WEIGHT_RANGE,
    vmp_range=VMP_RANGE,
)
GAE_LOAD_ENVS = corpus["corpus_name"]
CORPUS_DIR = Path(corpus["corpus_dir"])
train_ids = corpus["train_ids"]
val_ids = corpus["validation_ids"]
test_ids = corpus["test_ids"]

# Fixed split
TRAIN_ENV_ID = int(train_ids[0])
EVAL_ENV_IDS = [int(env_id) for env_id in corpus["graphs"]["id"] if int(env_id) != TRAIN_ENV_ID]
if TRAIN_ENV_ID not in set(map(int, corpus["graphs"]["id"])):
    raise ValueError(f"Training environment ID {TRAIN_ENV_ID} is not in the generated corpus.")

print("Fixed training scenario:", TRAIN_ENV_ID)
print("Held-out evaluation scenarios:", len(EVAL_ENV_IDS))
display(corpus["graphs"].head(10))
display(corpus["graphs"].describe(include="all"))

## 2. Discrete PPO without padding: inspect the space contract

Train ordinary discrete PPO on one scenario, then compare its spaces with a held-out scenario. On variable-size selected benchmark graphs this typically exposes a mismatch; VMP intentionally keeps its problem dimensions fixed, so its spaces should remain compatible across randomized scenarios. This diagnostic uses the same code either way.


In [ ]:
DISCRETE_CONFIGS = [
    {
        "name": "ppo_default",
        "total_timesteps": 25000,
        "kwargs": {
            "learning_rate": 0.001,
            "n_steps": 2048,
            "batch_size": 64,
            "ent_coef": 0.01,
        }
    },
    {
        "name": "ppo_higher_lr",
        "total_timesteps": 25000,
        "kwargs": {
            "learning_rate": 0.01,
            "n_steps": 2048,
            "batch_size": 64,
            "ent_coef": 0.01,
        }
    }
]

discrete_runs = {}
benchmark_results = {}

for cfg in DISCRETE_CONFIGS:
    label = f"discrete PPO / {cfg['name']}"
    paths = create_run_layout(LOG_ROOT, "discrete_ppo", environment=BENCHMARK)
    run_dir = paths["run"]
    print(f"\n=== Training {label} on corpus environment {TRAIN_ENV_ID} ===")
    training_env = load_benchmark_env(CORPUS_DIR, TRAIN_ENV_ID, BENCHMARK, gymnasium_api=True)
    run = train_agent_on_env(
        env=training_env,
        environment=BENCHMARK,
        algorithm="ppo",
        agent_type="discrete",
        total_timesteps=cfg["total_timesteps"],
        seed=SEED,
        algorithm_kwargs=cfg["kwargs"],
        save_path=str(paths["checkpoints"] / "model"),
        tensorboard_log=str(paths["tensorboard"]),
        verbose=1,
    )
    discrete_runs[cfg["name"]] = run
    save_agent_artifact(run["model"], f"discrete_ppo_{cfg['name']}", run_dir)
    save_training_metadata(
        run_dir, agent_type="discrete_ppo", environment=BENCHMARK, algorithm="ppo",
        total_timesteps=cfg["total_timesteps"], seed=SEED,
    )
    train_metric_frame = plot_tensorboard(paths["tensorboard"], title=f"Discrete PPO — {cfg['name']}")
    save_metric_frame(train_metric_frame, paths["train_metrics"])

## 2.1. Discrete PPO Evaluation

This cell discusses the evaluation of discrete PPO agents trained in Section 2, particularly targeting observation-action space mismatch.

In [ ]:
# Pick a held-out graph with a node count farthest from the training graph.
train_nodes = int(corpus["graphs"].loc[corpus["graphs"]["id"] == TRAIN_ENV_ID, "nodes"].iloc[0])
heldout_candidates = corpus["graphs"][corpus["graphs"]["id"] != TRAIN_ENV_ID].copy()
heldout_candidates["distance_from_train_size"] = (heldout_candidates["nodes"] - train_nodes).abs()
MISMATCH_ENV_ID = int(
    heldout_candidates.sort_values("distance_from_train_size", ascending=False).iloc[0]["id"]
)

raw_train_env = load_benchmark_env(CORPUS_DIR, TRAIN_ENV_ID, BENCHMARK, gymnasium_api=True)
raw_heldout_env = load_benchmark_env(CORPUS_DIR, MISMATCH_ENV_ID, BENCHMARK, gymnasium_api=True)
print(f"Training env {TRAIN_ENV_ID}: nodes={train_nodes}")
print(
    f"Held-out env {MISMATCH_ENV_ID}: nodes="
    f"{int(corpus['graphs'].loc[corpus['graphs']['id'] == MISMATCH_ENV_ID, 'nodes'].iloc[0])}"
)
print("\nRaw observation space (train):", raw_train_env.observation_space)
print("Raw observation space (held-out):", raw_heldout_env.observation_space)
print("Raw action space (train):", raw_train_env.action_space)
print("Raw action space (held-out):", raw_heldout_env.action_space)

same_obs_space = raw_train_env.observation_space == raw_heldout_env.observation_space
same_action_space = raw_train_env.action_space == raw_heldout_env.action_space
print(f"\nObservation spaces equal: {same_obs_space}")
print(f"Action spaces equal: {same_action_space}")

naive_model = discrete_runs[DISCRETE_CONFIGS[0]["name"]]["model"]

# Stable-Baselines3 checks both observation and action spaces when attaching an
# environment to a trained model. This is the clearest compatibility check.
try:
    naive_model.set_env(DummyVecEnv([lambda: load_benchmark_env(CORPUS_DIR, MISMATCH_ENV_ID, BENCHMARK, gymnasium_api=True)]))
except Exception as exc:
    print("SB3 refuses to attach the differently-sized environment to the unpadded model:")
    print(f"{type(exc).__name__}: {exc}")

heldout_obs, _ = raw_heldout_env.reset(seed=SEED + 1)
try:
    predicted_action, _ = naive_model.predict(heldout_obs, deterministic=True)
    print("Unpadded model.predict() returned:", predicted_action)
    if hasattr(raw_heldout_env.action_space, "n"):
        scalar_action = int(np.asarray(predicted_action).reshape(-1)[0])
        if scalar_action >= raw_heldout_env.action_space.n:
            print(
                f"Expected transfer problem: action {scalar_action} is outside the "
                f"held-out Discrete({raw_heldout_env.action_space.n}) space."
            )
        else:
            print(
                "predict() did not raise for this observation, but that alone does not "
                "make the policy compatible: its input/action contracts were built for "
                "the training spaces. Compare the spaces above and try other graph sizes."
            )
except Exception as exc:
    print("As expected, the unpadded policy cannot consume this observation:")
    print(f"{type(exc).__name__}: {exc}")

print(
    "\nConclusion: standard SB3 policies require a stable observation/action-space "
    "contract. Padding can make dimensions compatible; a graph encoder can additionally "
    "turn variable-size graph structure into a fixed-size learned observation."
)

## 3. Train the reference GAE/VGAE encoder

The GAE/VGAE trainer uses the corpus created once at the beginning.

In [ ]:
GAE_ENVIRONMENT = BENCHMARK
GAE_MODEL_TYPE = "gae"  # "gae" or "vgae"
GAE_TRAIN_ITERATIONS = 2500

gae_paths = create_run_layout(LOG_ROOT, "gae", environment=GAE_ENVIRONMENT)
gae_run = train_gae_for_embeddings(
    environment=GAE_ENVIRONMENT,
    load_envs=GAE_LOAD_ENVS,
    project_root=str(PROJECT_ROOT),
    train_iterations=GAE_TRAIN_ITERATIONS,
    model_type=GAE_MODEL_TYPE,
    seed=SEED,
    num_runs=1,
    validation=False,
    name="notebook_reference",
    verbose=1,
    compare_random_init=False,
)
gae_run_dir = Path(gae_run["output_dir"])

# Preserve the native trainer output under checkpoints, and mirror event files to tensorboard/.
shutil.copytree(gae_run_dir, gae_paths["checkpoints"] / "gae_bundle", dirs_exist_ok=True)
event_files = list(gae_run_dir.rglob("events.out.tfevents.*"))
for event_file in event_files:
    relative = event_file.relative_to(gae_run_dir)
    destination = gae_paths["tensorboard"] / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(event_file, destination)

save_training_metadata(
    gae_paths["run"], agent_type="gae", environment=GAE_ENVIRONMENT,
    algorithm=GAE_MODEL_TYPE, total_timesteps=GAE_TRAIN_ITERATIONS, seed=SEED,
)
if event_files:
    if BENCHMARK == "maxcut":
        gae_metric_frame = plot_tensorboard(gae_paths["tensorboard"], title="GAE training", metric="train/graph/total_loss")
        save_metric_frame(gae_metric_frame, gae_paths["train_metrics"], "gae_training_metrics.csv")
    elif BENCHMARK == "vmp":
        gae_metric_frame = plot_tensorboard(gae_paths["tensorboard"], title="GAE training (allocation)", metric="train/alloc_graph/total_loss")
        save_metric_frame(gae_metric_frame, gae_paths["train_metrics"], "gae_alloc_training_metrics.csv")
        gae_metric_frame = plot_tensorboard(gae_paths["tensorboard"], title="GAE training (latency)", metric="train/latency_graph/total_loss")
        save_metric_frame(gae_metric_frame, gae_paths["train_metrics"], "gae_latency_training_metrics.csv")
        gae_metric_frame = plot_tensorboard(gae_paths["tensorboard"], title="GAE training (traffic)", metric="train/traffic_graph/total_loss")
        save_metric_frame(gae_metric_frame, gae_paths["train_metrics"], "gae_traffic_training_metrics.csv")
else:
    print("No TensorBoard event files found in the native GAE output; checkpoints/configs were still copied.")

gae_bundle = load_gae_encoders(
    str(gae_run_dir),
    project_root=str(PROJECT_ROOT),
    prefer_best_validation=True,
)

## 4. Configure GAE-backed wrappers

The GAE bundle is shared by the GNN-discrete, iterative, and projection agents.

In [ ]:
WRAPPER_LOG_ROOT = gae_paths["run"] / "train_metrics" / "wrapper_logs"
iterative_wrapper_factory = make_agent_training_wrapper_factory(
    environment=BENCHMARK,
    gae_run_dir=str(gae_run_dir),
    agent_type="iterative",
    project_root=str(PROJECT_ROOT),
    logs_dir=str(WRAPPER_LOG_ROOT / "iterative_wrapper"),
)
projection_wrapper_factory = make_agent_training_wrapper_factory(
    environment=BENCHMARK,
    gae_run_dir=str(gae_run_dir),
    agent_type="projection",
    algorithm="ppo",
    project_root=str(PROJECT_ROOT),
    logs_dir=str(WRAPPER_LOG_ROOT / "projection_wrapper"),
)
print("Configured iterative and projection wrapper factories.")

## 5. Fix the mismatch: GNN observations + native space padding

The raw discrete baseline above uses the environment's size-dependent spaces directly. Here we follow the same mechanism as the CLI: wrap each corpus environment with `AgentTrainingWrapper` so the GNN supplies a learned, fixed-size observation representation, then let `GraphEnvSwitcher(pad_spaces=True)` compute the maximum action-space shape across the corpus and expose one consistent space to PPO.

In [ ]:

import sys
import importlib

# The canonical module used to instantiate your GNN models
canonical_model = importlib.import_module("LaGCORL.gae.model")

GAE = canonical_model.GAEEncoder
VGAE = canonical_model.VGAEEncoder

print("Canonical GAE:", GAE)

# Find EVERY loaded copy of agent_training_wrapper
patched_modules = []

for module_name, module in list(sys.modules.items()):

    if module is None:
        continue

    if module_name.endswith("agent_training_wrapper"):

        print("Patching module:", module_name)

        # Patch module-level references
        module.GAEEncoder = GAE
        module.VGAEEncoder = VGAE

        # Patch the actual encode function's global namespace
        wrapper_class = getattr(module, "AgentTrainingWrapper", None)

        if wrapper_class is not None:

            encode_function = wrapper_class.encode
            encode_globals = encode_function.__globals__

            encode_globals["GAEEncoder"] = GAE
            encode_globals["VGAEEncoder"] = VGAE

            print(
                "  GAE identity:",
                encode_globals["GAEEncoder"] is GAE
            )

            patched_modules.append(module_name)

# Normalize the duplicate model module reference
sys.modules["gae.model"] = canonical_model

if "gae" in sys.modules:
    sys.modules["gae"].model = canonical_model

print("\nPatched modules:", patched_modules)
print("Number of patched modules:", len(patched_modules))

assert len(patched_modules) > 0
print("\nSUCCESS: all discovered wrapper modules patched.")


In [ ]:
# The helper reproduces the CLI preprocessing: wrap each graph, serialize it,
# then build a GraphEnvSwitcher with one training graph and shared padding.
gnn_discrete_paths = create_run_layout(
    LOG_ROOT,
    "discrete_ppo_gnn_padded",
    environment=BENCHMARK
)

gnn_discrete_paths = create_run_layout(LOG_ROOT, "discrete_ppo_gnn_padded", environment=BENCHMARK)
gnn_discrete_run_dir = gnn_discrete_paths["run"]

discrete_gnn_wrapper_factory = make_agent_training_wrapper_factory(
    environment=BENCHMARK,
    gae_run_dir=str(gae_run_dir),
    agent_type="discrete",
    algorithm="ppo",
    project_root=str(PROJECT_ROOT),
    logs_dir=str(gae_paths["run"] / "train_metrics" / "wrapper_logs" / "discrete_gnn"),
)

padded_setup = build_padded_gnn_discrete_corpus(
    project_root=PROJECT_ROOT,
    corpus_dir=CORPUS_DIR,
    env_ids=corpus["graphs"]["id"].tolist(),
    train_env_id=TRAIN_ENV_ID,
    wrapper_factory=discrete_gnn_wrapper_factory,
    log_root=LOG_ROOT,
    run_dir=gnn_discrete_run_dir,
    environment=BENCHMARK,
)
padded_training_env = padded_setup["env"]
WRAPPED_CORPUS_DIR = padded_setup["wrapped_corpus_dir"]
padding_config = padded_setup["padding_config"]
print("Wrapped corpus:", WRAPPED_CORPUS_DIR.resolve())
print("GNN observation space:", padded_training_env.observation_space)
print("Padded action space:", padded_training_env.action_space)
print("Shared padding configuration:", padding_config)

gnn_discrete_run = train_agent_on_env(
    env=padded_training_env,
    environment=BENCHMARK,
    algorithm="ppo",
    agent_type="discrete",
    total_timesteps=25000,
    seed=SEED,
    algorithm_kwargs={
        "learning_rate": 0.001,
        "n_steps": 2048,
        "batch_size": 64,
        "ent_coef": 0.01,
    },
    save_path=str(gnn_discrete_paths["checkpoints"] / "model"),
    tensorboard_log=str(gnn_discrete_paths["tensorboard"]),
    verbose=1,
)
save_agent_artifact(gnn_discrete_run["model"], "discrete_ppo_gnn_padded", gnn_discrete_run_dir)
save_training_metadata(
    gnn_discrete_run_dir,
    agent_type="discrete_ppo_gnn_padded",
    environment=BENCHMARK,
    algorithm="ppo",
    total_timesteps=25000,
    seed=SEED,
)
gnn_discrete_metric_frame = plot_tensorboard(
    gnn_discrete_paths["tensorboard"],
    title="Discrete PPO with GNN observations and padding",
)
save_metric_frame(gnn_discrete_metric_frame, gnn_discrete_paths["train_metrics"])



## 5.1 Evaluation of the GNN-Padded Discrete PPO

In [ ]:
# Evaluate on every held-out graph using the exact padding configuration computed above.
EVAL_EPISODES = 20
EVAL_SEEDS = [101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120]

gnn_discrete_episodes = pd.concat(
    [
        evaluate_padded_gnn_policy(
            gnn_discrete_run["model"],
            env_id=env_id,
            train_env_id=TRAIN_ENV_ID,
            wrapped_corpus_dir=WRAPPED_CORPUS_DIR,
            padding_config=padding_config,
            log_dir=gnn_discrete_run_dir,
            n_episodes=EVAL_EPISODES,
            seeds=EVAL_SEEDS,
        )
        for env_id in EVAL_ENV_IDS
    ],
    ignore_index=True,
)
gnn_discrete_summary = gnn_discrete_episodes[
    ["episode_reward", "episode_length"]
].mean().to_dict()
gnn_discrete_result = {
    "episodes": gnn_discrete_episodes,
    "summary": gnn_discrete_summary,
}
benchmark_results["discrete PPO (GNN + padding)"] = gnn_discrete_result
gnn_discrete_episodes.to_csv(
    gnn_discrete_paths["test_metrics"] / "heldout_episode_metrics.csv",
    index=False,
)
print("Padded GNN discrete PPO held-out summary:", gnn_discrete_summary)

In [ ]:
display(gnn_discrete_episodes)

## 6. Train the iterative agent (using DQN)


In [ ]:
iterative_paths = create_run_layout(LOG_ROOT, "iterative_dqn", environment=BENCHMARK)
iterative_run_dir = iterative_paths["run"]
iterative_training_env = load_benchmark_env(CORPUS_DIR, TRAIN_ENV_ID, BENCHMARK)

iterative_run = train_agent_on_env(
    env=iterative_training_env,
    environment=BENCHMARK,
    algorithm="idqn",
    agent_type="iterative",
    total_timesteps=25000,
    seed=SEED,
    wrapper_factory=iterative_wrapper_factory,
    algorithm_kwargs={
        "learning_rate": 0.00001,
        "batch_size": 32,
        "exploration_fraction": 0.8,
        "exploration_initial_eps": 1.0,
        "exploration_final_eps": 0.05,
    },
    save_path=str(iterative_paths["checkpoints"] / "model"),
    tensorboard_log=str(iterative_paths["tensorboard"]),
    verbose=1,
)
save_agent_artifact(iterative_run["model"], "iterative_idqn", iterative_run_dir)
save_training_metadata(
    iterative_run_dir, agent_type="iterative_dqn", environment=BENCHMARK,
    algorithm="idqn", total_timesteps=25000, seed=SEED,
)
iterative_metric_frame = plot_tensorboard(iterative_paths["tensorboard"], title="Iterative iDQN — ep_rew_mean")
save_metric_frame(iterative_metric_frame, iterative_paths["train_metrics"])

## 6.1. Evaluate the iterative DQN

In [ ]:
iterative_result = benchmark_on_corpus(
    iterative_run["model"],
    corpus_dir=CORPUS_DIR,
    env_ids=EVAL_ENV_IDS,
    test_agent_on_env=test_agent_on_env,
    n_episodes=EVAL_EPISODES,
    seeds=EVAL_SEEDS,
    wrapper_factory=iterative_wrapper_factory,
    test_metrics_dir=iterative_paths["test_metrics"],
    label="iterative iDQN",
    verbose=1,
)
benchmark_results["iterative iDQN"] = iterative_result
display(iterative_result["episodes"].head())
print("Mean metrics across held-out environments:", iterative_result["summary"])

## 7. Train the projection PPO agent

In [ ]:
projection_paths = create_run_layout(LOG_ROOT, "projection_ppo", environment=BENCHMARK)
projection_run_dir = projection_paths["run"]
projection_training_env = load_benchmark_env(CORPUS_DIR, TRAIN_ENV_ID, BENCHMARK)

projection_run = train_agent_on_env(
    env=projection_training_env,
    environment=BENCHMARK,
    algorithm="ppo",
    agent_type="projection",
    total_timesteps=25000,
    seed=SEED,
    wrapper_factory=projection_wrapper_factory,
    algorithm_kwargs={
        "learning_rate": 0.0001,
        "n_steps": 2048,
        "batch_size": 128,
        "ent_coef": 0.01,
    },
    save_path=str(projection_paths["checkpoints"] / "model"),
    tensorboard_log=str(projection_paths["tensorboard"]),
    verbose=1,
)
save_agent_artifact(projection_run["model"], "projection_ppo", projection_run_dir)
save_training_metadata(
    projection_run_dir, agent_type="projection_ppo", environment=BENCHMARK,
    algorithm="ppo", total_timesteps=25000, seed=SEED,
)
projection_metric_frame = plot_tensorboard(projection_paths["tensorboard"], title="Projection PPO — ep_rew_mean")
save_metric_frame(projection_metric_frame, projection_paths["train_metrics"])


## 7.1. Evaluate the projection agent

In [ ]:
projection_result = benchmark_on_corpus(
    projection_run["model"],
    corpus_dir=CORPUS_DIR,
    env_ids=EVAL_ENV_IDS,
    test_agent_on_env=test_agent_on_env,
    n_episodes=EVAL_EPISODES,
    seeds=EVAL_SEEDS,
    wrapper_factory=projection_wrapper_factory,
    test_metrics_dir=projection_paths["test_metrics"],
    label="projection PPO",
    verbose=1,
)
benchmark_results["projection PPO"] = projection_result
display(projection_result["episodes"].head())
print("Mean metrics across held-out environments:", projection_result["summary"])

## 8. Final comparison: held-out domain-randomized environments

The padded GNN discrete PPO, iterative iDQN, and projection PPO agents are evaluated on the same held-out environment IDs (`1..N-1`). The raw unpadded discrete PPO model is deliberately excluded from the aggregate table because it is the negative example: its observation/action spaces are tied to the training scenario.

In [ ]:
# Cell 29: Compare all agents using shared metrics.
# Normalize metric names and ignore episode_reward / episode_length.

agent_frames = {
    label: result["episodes"].copy()
    for label, result in benchmark_results.items()
}

if BENCHMARK == "maxcut":
    # Metrics to compare (canonical names).
    expected_metrics = [
        "agent_current_cut",
        "agent_relative_performance",
        "agent_partition0_size",
        "agent_partition1_size",
        "agent_num_steps",
    ]
    
    # Map metric names that may appear without the "agent_" prefix.
    metric_aliases = {
        "current_cut": "agent_current_cut",
        "relative_performance": "agent_relative_performance",
        "partition0_size": "agent_partition0_size",
        "partition1_size": "agent_partition1_size",
        "num_steps": "agent_num_steps",
    }
elif BENCHMARK == "vmp":
    expected_metrics = [
        "agent_weighted_sum_difference",
        "agent_norm_avg_util_difference",
        "agent_norm_avg_power_difference",
        "agent_norm_avg_security_risk_difference",
        "agent_norm_avg_load_difference",
        "agent_norm_packing_efficiency_difference",
        "agent_vm_density_difference",
        "agent_active_pms_difference",
        "agent_actual_valid_movements",
    ]
    
    metric_aliases = {
        "weighted_sum_difference": "agent_weighted_sum_difference",
        "norm_avg_util_difference": "agent_norm_avg_util_difference",
        "norm_avg_power_difference": "agent_norm_avg_power_difference",
        "norm_avg_security_risk_difference": "agent_norm_avg_security_risk_difference",
        "norm_avg_load_difference": "agent_norm_avg_load_difference",
        "norm_packing_efficiency_difference": "agent_norm_packing_efficiency_difference",
        "vm_density_difference": "agent_vm_density_difference",
        "active_pms_difference": "agent_active_pms_difference",
        "actual_valid_movements": "agent_actual_valid_movements",
    }
else:
    raise ValueError(f"Unknown benchmark: {BENCHMARK}")

# Rename aliases to canonical names when the canonical name is absent.
for label, frame in agent_frames.items():
    rename_map = {
        alias: canonical
        for alias, canonical in metric_aliases.items()
        if alias in frame.columns and canonical not in frame.columns
    }
    agent_frames[label] = frame.rename(columns=rename_map)

# Warn if an agent is missing any comparison metric.
missing = {
    label: [
        metric for metric in expected_metrics
        if metric not in frame.columns
    ]
    for label, frame in agent_frames.items()
}

for label, absent in missing.items():
    if absent:
        print(f"WARNING — {label} is missing: {absent}")

# Keep only metrics available for every agent.
common_metrics = [
    metric
    for metric in expected_metrics
    if all(metric in frame.columns for frame in agent_frames.values())
]

if not common_metrics:
    raise ValueError(
        "No common evaluation metrics exist. Check the metric names "
        "in each agent's episode results."
    )

comparison_table = pd.DataFrame({
    label: frame[common_metrics].mean(numeric_only=True)
    for label, frame in agent_frames.items()
}).T

comparison_table.index.name = "agent"

print("Metrics used for comparison:", common_metrics)
display(comparison_table)

# Save combined episode metrics using the original column names.
all_episodes = pd.concat(
    [result["episodes"].copy() for result in benchmark_results.values()],
    ignore_index=True,
    sort=False,
)

comparison_csv = (
    LOG_ROOT
    / "comparison"
    / f"{BENCHMARK}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    / "episode_metrics_all_agents.csv"
)

comparison_csv.parent.mkdir(parents=True, exist_ok=True)
all_episodes.to_csv(comparison_csv, index=False)

print("Saved combined held-out episode metrics to:", comparison_csv.resolve())